# REL301m Assignment 2 — ADBE: 8 năm train / 2 năm test
Train 05/12/2008–05/12/2016, test 05/12/2016–05/12/2018, không trùng ngày. Vốn ban đầu 100.000 USD.
Notebook chỉ load/predict model đã lưu, không train lại. Mở thư mục project rồi Run All.

In [1]:
import hashlib, json
from pathlib import Path
import pandas as pd
from adbe_rl.config import load_config, OUTPUT_DIR
from adbe_rl.market import load_track
from adbe_rl.env import TradingEnv
from adbe_rl.policies import load_policy

config = load_config()
track = load_track(config, 'consecutive')   # kiểm tra 8 năm / 2 năm, không trùng ngày
print(json.dumps(track.describe(), indent=2))

{
  "fit": {
    "first_session": "2008-12-05",
    "last_session": "2015-12-04",
    "sessions": 1762
  },
  "validation": {
    "first_session": "2015-12-07",
    "last_session": "2016-12-02",
    "sessions": 251
  },
  "train": {
    "first_session": "2008-12-05",
    "last_session": "2016-12-02",
    "sessions": 2013
  },
  "test": {
    "first_session": "2016-12-05",
    "last_session": "2018-12-04",
    "sessions": 504
  },
  "gap_calendar_days": 0
}


## Bonus: load model đã lưu và predict trên 2 năm test
Mỗi test bắt đầu từ 100.000 USD tiền mặt. Episode dừng ngay khi thắng hoặc thua.

In [2]:
path = OUTPUT_DIR / 'consecutive/models/cem_temporal.json'
hash_before = hashlib.sha256(path.read_bytes()).hexdigest()
model = load_policy(path, config.rules)
env = TradingEnv(track.market, *track.ranges['test'], config.rules)
obs, info = env.reset()
while not env.done:
    action = model.predict(obs, env.account, env.action_mask())
    obs, reward, terminated, truncated, info = env.step(action)

print('Initial cash:     ', config.rules.initial_cash)
print('Test total assets:', round(info['assets'], 2))
print('Cumulative reward:', round(env.reward_sum, 6))
print('Stop date:        ', info['Date'], '| reason:', info['reason'])
assert hashlib.sha256(path.read_bytes()).hexdigest() == hash_before
summary = pd.read_csv(OUTPUT_DIR / 'consecutive/summary.csv', float_precision='round_trip')
stored = summary.query("model == 'cem_temporal' and period == 'test' and scenario == 'sell_high'").iloc[0]
assert info['assets'] == stored.final_assets
print('PASS: replay equals stored result; model file unchanged.')

Initial cash:      100000.0
Test total assets: 233742.79
Cumulative reward: 0.849051
Stop date:         2018-12-04 | reason: end_of_data
PASS: replay equals stored result; model file unchanged.


## Kết quả tất cả model (bán theo giá High)

In [3]:
columns = ['model', 'period', 'peak_assets', 'final_assets', 'cumulative_reward', 'end_reason']
print(summary[summary.scenario == 'sell_high'][columns].to_string(index=False))

                    model period   peak_assets  final_assets  cumulative_reward  end_reason
       penalty_aware_hold  train 428748.904930 382862.450890           1.342506 end_of_data
           cem_validation  train 460231.874818 415186.679816           1.423558 end_of_data
           cem_train_peak  train 463271.935404 417658.622808           1.429494 end_of_data
             cem_temporal  train 860523.739536 770544.617674           2.041927 end_of_data
cem_temporal_continuation  train 860523.739536 770544.617674           2.041927 end_of_data
       penalty_aware_hold   test 253645.537403 230625.490481           0.835625 end_of_data
           cem_validation   test 249773.477829 230503.998603           0.835098 end_of_data
           cem_train_peak   test 255494.330695 234553.068814           0.852512 end_of_data
             cem_temporal   test 254241.926029 233742.791998           0.849051 end_of_data
cem_temporal_continuation   test 254241.926029 233742.791998           0.849051 

## Kiểm chứng
Luật giao dịch, kế toán từng phiên, chia dữ liệu, tính nhân quả và replay model. Huấn luyện lại: `python main.py train`.

In [4]:
from adbe_rl.verify import verify
_ = verify(config, OUTPUT_DIR)

[consecutive] verified
[dotcom] verified
{
  "status": "PASS",
  "checks": [
    "8-year train and 2-year test calendar windows, zero overlap, train before test",
    "buy orders within 35-73% of cash, sell orders within 13-82% of shares, whole shares",
    "4% fee on both sides; cash, shares and assets reconcile every session",
    "penalty exactly after 21 idle sessions; failed orders do not reset the counter",
    "buy fills at previous High, sell fills at today High/Close",
    "cumulative reward equals the sum of rewards; only the last row ends the episode",
    "observations unaffected by current or future prices",
    "single-episode env equals the vectorized simulator",
    "load/predict replay reproduces stored summaries",
    "model files unchanged by evaluation"
  ],
  "audited_histories": 80
}
